# Hyrax HATS Phase 2 Example with `HyraxLoopback`

This notebook demonstrates how to configure and exercise `HyraxHATSDataset` (phase-2 lazy, partition-aware path) using the built-in `HyraxLoopback` model.

It is designed so you can replace one path (`HATS_PATH`) with your own catalog (for example your ~7GB catalog) and iterate quickly.


## What this notebook demonstrates

1. Setting up a `data_request` that uses `HyraxHATSDataset`.
2. Phase-2 HATS options (`bundle_size`, `max_cached_bundles`, `project_columns`, `strict_metadata`).
3. Accessing rows/fields through `get_<field>(idx)`.
4. Observing bundle-cache behavior.
5. Running end-to-end `infer()` with `HyraxLoopback`.


In [ ]:
from pathlib import Path
import numpy as np

import hyrax

print('Hyrax version:', hyrax.__version__)


In [ ]:
# TODO: replace with your local HATS catalog path
HATS_PATH = Path('/path/to/your/hats_catalog')

if not HATS_PATH.exists():
    print(f'WARNING: {HATS_PATH} does not exist yet. Update HATS_PATH before running full notebook.')

# Adjust these to columns in your catalog
PRIMARY_ID_FIELD = 'object_id'
FEATURE_FIELDS = ['coord_ra', 'coord_dec']

print('Primary ID:', PRIMARY_ID_FIELD)
print('Feature fields:', FEATURE_FIELDS)


In [ ]:
h = hyrax.Hyrax()
h.set_config('model.name', 'HyraxLoopback')

# Optional, keeps data loading deterministic while exploring
h.set_config('data_loader.shuffle', False)

data_request = {
    'infer': {
        'catalog': {
            'dataset_class': 'HyraxHATSDataset',
            'data_location': str(HATS_PATH),
            'fields': FEATURE_FIELDS,
            'primary_id_field': PRIMARY_ID_FIELD,
            'split_fraction': 1.0,
            # Phase-2 options
            'hats': {
                'bundle_size': 256,
                'max_cached_bundles': 64,
                'project_columns': 'auto',   # 'auto' | 'all' | explicit list
                'strict_metadata': True,
            },
        }
    }
}

h.set_config('data_request', data_request)
data_request


In [ ]:
prepared = h.prepare()
dataset = prepared['infer']._primary_or_first_dataset()

print('Dataset class:', dataset.__class__.__name__)
print('Length:', len(dataset))
print('Columns (first 15):', dataset.column_names[:15])
print('Sample data keys:', list(dataset.sample_data()['data'].keys())[:10])


In [ ]:
# Demonstrate dynamic getters
first_idx = 0
primary_getter = getattr(dataset, f'get_{PRIMARY_ID_FIELD}')

print('ID at row 0:', primary_getter(first_idx))
for field in FEATURE_FIELDS:
    getter = getattr(dataset, f'get_{field}')
    print(f'{field} at row 0:', getter(first_idx))


In [ ]:
# Demonstrate bundle-cache effects via repeated nearby access
# (uses internal stats for exploration while phase-2 is maturing)
cache = dataset._accessor.cache
print('Initial cache hits/misses:', cache.hits, cache.misses)

for idx in [0, 1, 2, 2, 1, 0]:
    _ = getattr(dataset, f'get_{FEATURE_FIELDS[0]}')(idx)

print('Post-access cache hits/misses:', cache.hits, cache.misses)


In [ ]:
# Run end-to-end inference. HyraxLoopback returns model input values.
inference_results = h.infer()

print('Number of inference rows:', len(inference_results))
print('First 3 result ids:', inference_results.ids()[:3])

for i in range(3):
    print(f'Result {i}:', inference_results[i])


## Suggested experiments with your 7GB catalog

- Increase/decrease `bundle_size` and observe cache hit/miss behavior and wall time.
- Compare `project_columns='auto'` versus `'all'`.
- Try different `fields` lists in `data_request` to validate projected reads.
- Enable debug logging to see cache and catalog-open behavior in detail.
